# Riepilogo finale: MTGFlow, STGAN e SDE-Net

Questo notebook non ricalcola score o predizioni. Consolida esclusivamente gli output delle analisi spaziale e di sensitivity, mantenendo distinti detector, eventi e orizzonti.

In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
SPATIAL_DIR = Path(os.environ.get('SPATIAL_COMPARISON_OUT_DIR', ROOT / 'outputs/anomaly_spatial_comparison')).resolve()
SENSITIVITY_DIR = Path(os.environ.get('ANOMALY_SENSITIVITY_OUT_DIR', ROOT / 'outputs/anomaly_threshold_sensitivity_t1_t6')).resolve()
OUT_DIR = Path(os.environ.get('ANOMALY_SUMMARY_OUT_DIR', ROOT / 'outputs/anomaly_analysis_summary')).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
required = [
    SPATIAL_DIR / 'event_detector_metrics_by_location.csv',
    SPATIAL_DIR / 'event_forecast_metrics_by_location.csv',
    SENSITIVITY_DIR / 'reference_decision_metrics.csv',
]
missing = [path for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Eseguire prima i notebook spaziale e sensitivity:\n' + '\n'.join(map(str, missing)))

## 1. Estensione delle anomalie negli eventi

In [ ]:
detector = pd.read_csv(required[0])
event_detector = (detector.groupby(['detector', 'event'], observed=True)
    .agg(n_locations=('location', 'nunique'), n_observations=('n_observations', 'sum'), n_anomalies=('n_anomalies', 'sum'), mean_location_anomaly_fraction=('anomaly_fraction', 'mean'), max_location_anomaly_fraction=('anomaly_fraction', 'max'), max_intensity=('intensity_max', 'max')).reset_index())
event_detector['regional_anomaly_fraction'] = event_detector['n_anomalies'] / event_detector['n_observations']
event_detector.to_csv(OUT_DIR / 'event_detector_summary.csv', index=False)
display(event_detector)

## 2. Errore di forecasting per evento e orizzonte

In [ ]:
forecast = pd.read_csv(required[1])
event_forecast = (forecast.groupby(['event', 'horizon_hours'], observed=True)
    .agg(n_locations=('location', 'nunique'), n_forecasts=('n_forecasts', 'sum'), sum_error=('sum_error', 'sum'), sum_abs_error=('sum_abs_error', 'sum'), sum_squared_error=('sum_squared_error', 'sum')).reset_index())
event_forecast['bias'] = event_forecast['sum_error'] / event_forecast['n_forecasts']
event_forecast['mae'] = event_forecast['sum_abs_error'] / event_forecast['n_forecasts']
event_forecast['rmse'] = np.sqrt(event_forecast['sum_squared_error'] / event_forecast['n_forecasts'])
event_forecast.to_csv(OUT_DIR / 'event_forecast_summary.csv', index=False)
display(event_forecast)

## 3. Prestazioni alle decisioni di riferimento a priori

In [ ]:
references = pd.read_csv(required[2]).sort_values(['detector', 'horizon_hours'])
references.to_csv(OUT_DIR / 'reference_detector_forecast_summary.csv', index=False)
display(references[['detector', 'horizon_hours', 'decision_threshold', 'threshold_kind', 'rare_fraction', 'mae_normal', 'mae_rare', 'rmse_normal', 'rmse_rare']])

## 4. Figure principali

In [ ]:
figure_paths = [
    SENSITIVITY_DIR / 'figures/mtgflow_mae_rmse_sensitivity_t1_t6.png',
    SENSITIVITY_DIR / 'figures/stgan_mae_rmse_sensitivity_t1_t6.png',
    SENSITIVITY_DIR / 'figures/classification_sensitivity_mtgflow_stgan_t1_t6.png',
]
figure_paths += sorted((SPATIAL_DIR / 'figures').glob('*_spatial_comparison_regional.png'))
for path in figure_paths:
    if path.is_file(): display(Image(filename=str(path)))
print('Figure mostrate:', sum(path.is_file() for path in figure_paths))

In [ ]:
metadata = {
    'post_processing_only': True,
    'spatial_source': str(SPATIAL_DIR),
    'sensitivity_source': str(SENSITIVITY_DIR),
    'selection_warning': 'Thresholds must be selected on validation and frozen before the 2019 test.',
}
(OUT_DIR / 'summary_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Riepilogo:', OUT_DIR)